# Predictive Analytics Using Historical Data
**Data Analytics Internship Project**

**Goal:** Build a predictive model that forecasts future trends using historical data.

**Steps I followed:**
1. Load and explore the data
2. Clean and preprocess the data
3. Analyse the trend and seasonality
4. Build prediction models (regression, Holt-Winters and SARIMA time-series models)
5. Evaluate the models (MAE, RMSE, MAPE, R2) and visualize predictions
6. Forecast the next 12 months
7. Conclusion

## 1. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.optimize import minimize

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 5)

## 2. Loading the Data
The dataset has monthly sales of a retail store from 2015 to 2023.
It has two columns: `Date` and `Sales`.

In [ ]:
df = pd.read_csv("sales_data.csv")
print("Shape of data:", df.shape)
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Data Cleaning and Preprocessing

### 3.1 Checking missing values and duplicates

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())

There are some missing values in `Sales` and a few duplicate rows. I will fix both.

In [ ]:
# removing duplicate rows
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

# converting Date column to datetime and using it as index
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").set_index("Date")

# checking that every month is present (should be 108 months)
print("Number of months:", len(df))
print("Start:", df.index.min().date(), "| End:", df.index.max().date())

### 3.2 Checking outliers

In [ ]:
plt.figure()
plt.plot(df.index, df["Sales"], marker="o", markersize=3)
plt.title("Monthly Sales (before handling outliers)")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.show()

In the graph I can see two very high spikes that do not follow the pattern.
They look like data entry mistakes. To find them properly I compare each value with the rolling median
of its nearby months. If a value is more than 2 times the rolling median, I treat it as an outlier.

In [ ]:
rolling_median = df["Sales"].rolling(window=5, center=True, min_periods=1).median()
outliers = df["Sales"] > 2 * rolling_median

print("Outliers found:", outliers.sum())
print(df[outliers])

# replacing outliers with NaN so they get filled along with the missing values
df.loc[outliers, "Sales"] = np.nan

### 3.3 Filling missing values

In [ ]:
# time based interpolation is good for time series because it uses nearby months
df["Sales"] = df["Sales"].interpolate(method="time")

print("Missing values left:", df["Sales"].isnull().sum())

In [ ]:
plt.figure()
plt.plot(df.index, df["Sales"], color="green")
plt.title("Monthly Sales (after cleaning)")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.show()

## 4. Exploratory Data Analysis (Trend and Seasonality)

In [ ]:
# yearly total sales
yearly = df["Sales"].groupby(df.index.year).sum()

plt.figure(figsize=(9, 4))
plt.bar(yearly.index, yearly.values, color="steelblue")
plt.title("Total Sales per Year")
plt.xlabel("Year")
plt.ylabel("Total Sales")
plt.show()
yearly

Sales are going up every year, so there is a clear **upward trend**.

In [ ]:
# 12 month rolling average shows the trend clearly
plt.figure()
plt.plot(df["Sales"], label="Monthly sales", alpha=0.6)
plt.plot(df["Sales"].rolling(12).mean(), label="12 month moving average", color="red", linewidth=2)
plt.title("Sales Trend")
plt.legend()
plt.show()

In [ ]:
# seasonal index: each month's sales compared with the average of its own year
# (this removes the effect of the trend, so only the seasonal pattern is left)
yearly_avg = df["Sales"].groupby(df.index.year).transform("mean")
seasonal_index = (df["Sales"] / yearly_avg).groupby(df.index.month).mean()

plt.figure(figsize=(9, 4))
sns.barplot(x=seasonal_index.index, y=seasonal_index.values, color="orange")
plt.axhline(1, color="black", linestyle="--")
plt.title("Seasonal Index by Month (1.0 = average month)")
plt.xlabel("Month")
plt.ylabel("Seasonal Index")
plt.show()

**Observations:**
- There is a steady upward trend over the years.
- There is a **seasonal pattern**: sales are lowest in Jan-Feb (index below 1) and highest in Nov-Dec (index well above 1, festival / holiday season).

So the model should understand both the trend and the month of the year.

## 5. Feature Engineering

Models cannot use the date directly, so I create features from it:
- `time_index` – month number from the start (captures trend)
- `month` – month of the year (captures seasonality)
- `lag_1`, `lag_12` – sales of last month and same month last year

In [ ]:
data = df.copy()
data["time_index"] = np.arange(len(data))
data["month"] = data.index.month

# lag features (past values)
data["lag_1"] = data["Sales"].shift(1)
data["lag_12"] = data["Sales"].shift(12)

# first 12 rows will have NaN lag values, so drop them
data = data.dropna()
data.head()

## 6. Train and Test Split

For time series data I cannot split randomly, because that would let the model see the future.
So I use the **older data for training** and the **latest 24 months for testing**.

In [ ]:
test_months = 24

train = data.iloc[:-test_months]
test = data.iloc[-test_months:]

print("Training data:", train.index.min().date(), "to", train.index.max().date(), "->", len(train), "rows")
print("Testing data :", test.index.min().date(), "to", test.index.max().date(), "->", len(test), "rows")

## 7. Building the Models

I build five models and compare them:
1. **Baseline** – predict the same value as last year's same month (simple benchmark)
2. **Linear Regression** – using time index and month (one-hot encoded)
3. **Random Forest Regressor** – using time index, month and lag features
4. **Holt-Winters** (triple exponential smoothing) – a classic time-series model with level, trend and seasonality
5. **SARIMA** – a seasonal ARIMA time-series model (needs the `statsmodels` library)

In [ ]:
y_train = train["Sales"]
y_test = test["Sales"]

# ---- Model 1: baseline (same month last year) ----
pred_baseline = test["lag_12"]

# ---- Model 2: Linear Regression ----
def make_lr_features(d):
    X = pd.DataFrame({"time_index": d["time_index"].values}, index=d.index)
    month_dummies = pd.get_dummies(d["month"], prefix="m").astype(int)
    # make sure all 12 months columns exist
    for m in range(1, 13):
        if f"m_{m}" not in month_dummies.columns:
            month_dummies[f"m_{m}"] = 0
    month_dummies = month_dummies[[f"m_{m}" for m in range(1, 13)]]
    return pd.concat([X, month_dummies], axis=1)

X_train_lr = make_lr_features(train)
X_test_lr = make_lr_features(test)

lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train)
pred_lr = lr_model.predict(X_test_lr)

# ---- Model 3: Random Forest ----
features_rf = ["time_index", "month", "lag_1", "lag_12"]

rf_model = RandomForestRegressor(n_estimators=300, random_state=42)
rf_model.fit(train[features_rf], y_train)
pred_rf = rf_model.predict(test[features_rf])

print("Baseline, Linear Regression and Random Forest trained")

# dictionary to keep the test predictions of every model
predictions = {
    "Baseline (last year same month)": np.asarray(pred_baseline),
    "Linear Regression": np.asarray(pred_lr),
    "Random Forest": np.asarray(pred_rf),
}

### Model 4: Holt-Winters (Triple Exponential Smoothing)
Holt-Winters keeps three things updated every month: the **level**, the **trend** and the **seasonal factor** of each month.
It has three smoothing values (alpha, beta, gamma) between 0 and 1. Instead of guessing them, I let the computer find the
values that give the smallest error on the training data.
I use the multiplicative seasonality version because the seasonal swings get bigger as sales grow.

In [ ]:
def holt_winters(y, alpha, beta, gamma, m=12):
    # y = numpy array of sales, m = season length (12 months)
    n = len(y)
    level = y[:m].mean()
    trend = (y[m:2*m].mean() - y[:m].mean()) / m
    season = list(y[:m] / y[:m].mean())      # starting seasonal factors
    fitted = []
    for t in range(n):
        fitted.append((level + trend) * season[t])
        new_level = alpha * y[t] / season[t] + (1 - alpha) * (level + trend)
        new_trend = beta * (new_level - level) + (1 - beta) * trend
        season.append(gamma * y[t] / new_level + (1 - gamma) * season[t])
        level, trend = new_level, new_trend
    return np.array(fitted), level, trend, season

def hw_forecast(level, trend, season, n, steps, m=12):
    # n = number of data points used for fitting
    return np.array([(level + h * trend) * season[n + (h - 1) % m] for h in range(1, steps + 1)])

def hw_error(params, y):
    fitted, _, _, _ = holt_winters(y, *params)
    return np.sum((y - fitted) ** 2)

def fit_holt_winters(y):
    result = minimize(hw_error, x0=[0.3, 0.1, 0.3], args=(y,),
                      bounds=[(0.001, 0.999)] * 3, method="L-BFGS-B")
    return result.x

# the time series models use the full cleaned series, with the same last 24 months as test data
y_ts = df["Sales"].values
y_ts_train = y_ts[:-test_months]

hw_params = fit_holt_winters(y_ts_train)
print("Best alpha, beta, gamma:", hw_params.round(3))

_, hw_level, hw_trend, hw_season = holt_winters(y_ts_train, *hw_params)
pred_hw = hw_forecast(hw_level, hw_trend, hw_season, len(y_ts_train), test_months)
predictions["Holt-Winters"] = pred_hw

### Model 5: SARIMA
**ARIMA** models predict a value using its own past values (AR), differencing to remove the trend (I) and past errors (MA).
**SARIMA** adds the same idea for the seasonal pattern (every 12 months). It is written as
SARIMA(p, d, q)(P, D, Q, 12).

ARIMA models need **stationary** data (no trend or seasonality), so first I check this with the ADF test.

In [ ]:
# SARIMA needs the statsmodels library: pip install statsmodels
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    from statsmodels.tsa.stattools import adfuller
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print("statsmodels is not installed. Run: pip install statsmodels  (SARIMA part will be skipped)")

In [ ]:
if HAS_STATSMODELS:
    ts = df["Sales"].asfreq("MS")
    ts_train = ts.iloc[:-test_months]

    # ADF test: p-value below 0.05 means the data is stationary
    p_original = adfuller(ts_train)[1]
    ts_diff = ts_train.diff().diff(12).dropna()      # remove trend (d=1) and seasonality (D=1)
    p_diff = adfuller(ts_diff)[1]
    print("ADF p-value of original data      :", round(p_original, 4))
    print("ADF p-value after differencing    :", round(p_diff, 4))

    # ACF and PACF plots help to choose p, q, P, Q
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    plot_acf(ts_diff, lags=24, ax=axes[0])
    plot_pacf(ts_diff, lags=24, ax=axes[1])
    plt.show()

The original data is not stationary because of the trend and seasonality. After differencing once normally (d=1)
and once seasonally (D=1), it becomes stationary. The ACF/PACF plots show only small spikes, so a simple model
with order (1, 1, 1) and seasonal order (1, 1, 1, 12) is a good choice.

In [ ]:
if HAS_STATSMODELS:
    sarima_model = SARIMAX(ts_train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12),
                           enforce_stationarity=False, enforce_invertibility=False)
    sarima_fit = sarima_model.fit(disp=False)
    print(sarima_fit.summary().tables[1])

    sarima_result = sarima_fit.get_forecast(steps=test_months)
    pred_sarima = np.asarray(sarima_result.predicted_mean)
    sarima_ci = sarima_result.conf_int()
    predictions["SARIMA (1,1,1)(1,1,1,12)"] = pred_sarima
    print("SARIMA trained, AIC =", round(sarima_fit.aic, 2))

## 8. Model Evaluation
I use these metrics:
- **MAE** – average error (in sales units)
- **RMSE** – like MAE but punishes big errors more
- **MAPE** – average error in percentage
- **R2** – how much of the variation the model explains (closer to 1 is better)

In [ ]:
def evaluate(name, actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    r2 = r2_score(actual, predicted)
    return {"Model": name, "MAE": round(mae, 2), "RMSE": round(rmse, 2),
            "MAPE (%)": round(mape, 2), "R2": round(r2, 3)}

results = pd.DataFrame([evaluate(name, y_test, pred) for name, pred in predictions.items()])
results

In [ ]:
best = results.sort_values("RMSE").iloc[0]
print("Best model based on RMSE:", best["Model"])

### Actual vs Predicted

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train.index, y_train, label="Training data", color="gray", alpha=0.6)
plt.plot(test.index, y_test, label="Actual (test)", color="black", linewidth=2)
for name, pred in predictions.items():
    if name.startswith("Baseline"):
        continue
    plt.plot(test.index, pred, label=name, linestyle="--")
plt.title("Actual vs Predicted Sales")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.show()

In [ ]:
# zoomed view on the test period only
plt.figure(figsize=(12, 5))
plt.plot(test.index, y_test, label="Actual", marker="o", color="black")
for name, pred in predictions.items():
    if name.startswith("Baseline"):
        continue
    plt.plot(test.index, pred, label=name, marker="s", markersize=4)
plt.title("Test Period: Actual vs Predicted")
plt.legend()
plt.show()

In [ ]:
# SARIMA forecast with its 95% confidence interval
if HAS_STATSMODELS:
    plt.figure(figsize=(12, 5))
    plt.plot(ts_train.index[-36:], ts_train.values[-36:], label="Training data (last 3 years)", color="gray")
    plt.plot(test.index, y_test, label="Actual", color="black", linewidth=2)
    plt.plot(test.index, pred_sarima, label="SARIMA forecast", color="green")
    plt.fill_between(test.index, sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1], color="green", alpha=0.2, label="95% interval")
    plt.title("SARIMA Forecast on Test Period")
    plt.legend()
    plt.show()

In [ ]:
# comparing the error of all models
plt.figure(figsize=(10, 4))
sns.barplot(data=results.sort_values("RMSE"), x="RMSE", y="Model", color="slateblue")
plt.title("RMSE of All Models (lower is better)")
plt.show()

In [ ]:
# residuals (errors) of Linear Regression - should look random around zero
res_lr = y_test - pred_lr

plt.figure(figsize=(11, 4))
plt.bar(test.index, res_lr, width=20, color="teal")
plt.axhline(0, color="black")
plt.title("Residuals of Linear Regression (Actual - Predicted)")
plt.show()

In [ ]:
# which features were important for random forest?
importance = pd.Series(rf_model.feature_importances_, index=features_rf).sort_values()

plt.figure(figsize=(7, 3.5))
importance.plot(kind="barh", color="purple")
plt.title("Random Forest Feature Importance")
plt.show()

## 9. Forecasting the Next 12 Months
Now I retrain the models on **all** the available data and predict the future 12 months (2024).
I use Linear Regression and Holt-Winters (and SARIMA if statsmodels is installed), and compare their forecasts.

In [ ]:
# making future dates
last_date = df.index.max()
future_dates = pd.date_range(last_date + pd.offsets.MonthBegin(1), periods=12, freq="MS")
future = pd.DataFrame(index=future_dates)
future["time_index"] = np.arange(data["time_index"].max() + 1, data["time_index"].max() + 13)
future["month"] = future.index.month

# Linear Regression retrained on full data
final_lr = LinearRegression()
final_lr.fit(make_lr_features(data), data["Sales"])
future["Linear Regression"] = final_lr.predict(make_lr_features(future))

# Holt-Winters retrained on full data
hw_params_full = fit_holt_winters(y_ts)
_, lvl, trd, sea = holt_winters(y_ts, *hw_params_full)
future["Holt-Winters"] = hw_forecast(lvl, trd, sea, len(y_ts), 12)

# SARIMA retrained on full data
if HAS_STATSMODELS:
    final_sarima = SARIMAX(ts, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12),
                           enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    future["SARIMA"] = np.asarray(final_sarima.get_forecast(steps=12).predicted_mean)

forecast_cols = [c for c in future.columns if c not in ["time_index", "month"]]
future[forecast_cols].round(2)

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df.index[-48:], df["Sales"].values[-48:], label="Historical sales (last 4 years)", color="steelblue", linewidth=2)
for col in forecast_cols:
    plt.plot(future.index, future[col], marker="o", markersize=4, label=col + " forecast")
plt.title("Sales Forecast for Next 12 Months")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.show()

In [ ]:
print("Total forecast sales for 2024:")
for col in forecast_cols:
    print("  {:<18}: {:,.0f}".format(col, future[col].sum()))
print("Total actual sales in 2023   : {:,.0f}".format(df.loc["2023", "Sales"].sum()))

## 10. Conclusion

In [ ]:
print("Best model on test data :", best["Model"])
print("Its MAPE                :", best["MAPE (%)"], "%")
print("Its R2 score            :", best["R2"])
print("Models compared         :", len(results))

**Summary:**
- The data had missing values, duplicate rows and outliers, which I cleaned before modelling.
- The sales have a clear **upward trend** and a **yearly seasonal pattern** (peak in Nov-Dec).
- I compared a baseline, Linear Regression, Random Forest, Holt-Winters and SARIMA using a time based train/test split (last 24 months as test).
- The results table and charts above show which model has the lowest MAE, RMSE and MAPE and the highest R2. Models that understand both trend and seasonality (Linear Regression with month features, Holt-Winters, SARIMA) are expected to beat the baseline.
- Random Forest struggles with this data because tree based models cannot predict values higher than what they saw in training, so they handle a strong upward trend badly.
- Using the retrained models, I forecasted the next 12 months. The models give similar forecasts that keep the trend and the seasonal pattern.

**Limitations and future improvements:**
- Only the date was used. Extra data like marketing spend, discounts or holidays could improve accuracy.
- I can tune the SARIMA order automatically (for example with a grid search on AIC) or try Prophet.
- The forecast assumes the past trend continues, so it should be updated regularly with new data.

**Learnings:** predictive modelling, time series train/test splitting, trend and seasonality analysis,
and data-driven forecasting.